# Réseau de neurones MLP — Prédiction de l'inhibition de BSEP

Ce notebook complète le **SVM** déjà réalisé. Il utilise les mêmes molécules et des fingerprints **ECFP4 / Morgan (1 024 bits)** pour comparer un **réseau de neurones MLP** au SVM sur une classification binaire.

**Contexte biologique :** BSEP (codé par *ABCB11*) transporte les sels biliaires hors des hépatocytes. Une inhibition de BSEP peut contribuer à certaines lésions hépatiques médicamenteuses. La classe `active` prédit l'inhibition de BSEP, **pas l'ensemble de la toxicité du foie**.

**Objectifs :** construire `X` avec RDKit, séparer train/test sans fuite, optimiser un réseau de neurones, afficher les courbes d'apprentissage et évaluer ROC-AUC, rappel, précision, F1 et matrice de confusion.

## 1. Installer et importer les bibliothèques

Un MLP (*Multi-Layer Perceptron*) est un réseau de neurones à couches entièrement connectées. On utilise `MLPClassifier` de scikit-learn pour garder du code accessible.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("rdkit") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
import joblib

GRAINE = 42
SORTIE = Path("resultats_nn")
SORTIE.mkdir(exist_ok=True, parents=True)
print("Environnement prêt.")

## 2. Charger les données Morgan–Warner

Le dataset public contient 660 molécules. La cible est `active` (0 : non-inhibiteur ; 1 : inhibiteur). On ne donne **pas** `pIC50` au réseau : c'est une mesure expérimentale associée à la cible.

In [ ]:
URL = ("https://raw.githubusercontent.com/ATOMScience-org/AMPL/master/"
       "atomsci/ddm/examples/BSEP/data/morgan_warner_combined_bsep_data.csv")
chemins = [Path("data/morgan_warner_combined_bsep_data.csv"),
           Path("../data/morgan_warner_combined_bsep_data.csv")]
csv_local = next((p for p in chemins if p.exists()), None)
df = pd.read_csv(csv_local if csv_local is not None else URL)
print(df.shape)
display(df.head())
print(df["active"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(6, 4))
n = df["active"].value_counts().reindex([0,1], fill_value=0)
ax.bar(["Non-inhibiteurs", "Inhibiteurs"], n, color=["#72A4BD", "#D6A45E"])
ax.set_title("Répartition des classes BSEP")
ax.set_ylabel("Nombre de molécules")
fig.tight_layout()
fig.savefig(SORTIE / "01_classes.png", dpi=160)
plt.show()

## 3. SMILES → RDKit → fingerprints ECFP4

Le SMILES décrit une structure chimique sous forme de texte. RDKit reconstruit la molécule, puis calcule **1 024 valeurs binaires** à partir de motifs chimiques. Ce sont exactement les caractéristiques utilisées pour le SVM. **Aucune tokenisation de SMILES n'est nécessaire.**

In [ ]:
d = df.dropna(subset=["base_rdkit_smiles", "active"]).copy()
d["active"] = pd.to_numeric(d["active"], errors="coerce")
d = d[d["active"].isin([0,1])].copy()
d["active"] = d["active"].astype(int)

def canon(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True) if mol else None

d["canonical_smiles"] = d["base_rdkit_smiles"].apply(canon)
d = d.dropna(subset=["canonical_smiles"])
conflits = d.groupby("canonical_smiles")["active"].nunique()
d = d[~d["canonical_smiles"].isin(conflits[conflits > 1].index)]
d = d.drop_duplicates("canonical_smiles").reset_index(drop=True)
generateur = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
X = np.asarray([generateur.GetFingerprintAsNumPy(Chem.MolFromSmiles(s))
                for s in d["canonical_smiles"]], dtype=np.float32)
y = d["active"].to_numpy()
print("Molécules :", len(d), " | X :", X.shape, " | y :", y.shape)
assert X.shape[1] == 1024

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(X[:30,:100], cmap="Blues", aspect="auto", interpolation="nearest")
ax.set_title("30 molécules × 100 premiers bits ECFP4")
ax.set_xlabel("Bit")
ax.set_ylabel("Molécule")
fig.tight_layout()
fig.savefig(SORTIE / "02_fingerprints.png", dpi=160)
plt.show()

## 4. Garder le même découpage train/test que le SVM

On teste sur **20 % des molécules**, sans utiliser ce jeu pour sélectionner le meilleur réseau. Si le fichier `data/splits_bsep.csv` est disponible, il est prioritaire ; sinon, on reproduit le split initial stratifié avec la graine 42.

In [ ]:
fichiers_split = [Path("data/splits_bsep.csv"), Path("../data/splits_bsep.csv"),
                  Path("resultats_svm/splits_bsep.csv")]
f_split = next((p for p in fichiers_split if p.exists()), None)
if f_split is None:
    i_train, i_test = train_test_split(np.arange(len(y)), test_size=.20,
                                      stratify=y, random_state=GRAINE)
else:
    separations = pd.read_csv(f_split)
    separations["canonical_smiles"] = separations["canonical_smiles"].apply(canon)
    assert not separations["canonical_smiles"].duplicated().any()
    repartition = d["canonical_smiles"].map(
        separations.set_index("canonical_smiles")["split"])
    assert repartition.notna().all(), "Des molécules sont absentes du split fourni."
    assert repartition.isin(["train", "test"]).all()
    i_train = np.flatnonzero(repartition.to_numpy() == "train")
    i_test = np.flatnonzero(repartition.to_numpy() == "test")

X_train, X_test = X[i_train], X[i_test]
y_train, y_test = y[i_train], y[i_test]
assert len(set(i_train).intersection(i_test)) == 0
assert len(np.unique(y_train)) == len(np.unique(y_test)) == 2
print("Train :", len(y_train), " | Test :", len(y_test))
split_export = d[["canonical_smiles", "active"]].copy()
split_export["split"] = "train"
split_export.loc[i_test, "split"] = "test"
split_export.to_csv(SORTIE / "splits_bsep.csv", index=False)

## 5. Comprendre l'architecture du MLP

**1 024 bits d'entrée → couches cachées (64 neurones ou 128 puis 32 neurones) → une probabilité d'inhibition.**

La fonction **ReLU** introduit des non-linéarités. L'optimiseur **Adam** modifie les poids. `alpha` ajoute une régularisation L2. Avec `early_stopping=True`, une petite fraction de l'entraînement sert à arrêter le réseau quand il cesse de progresser, ce qui limite le surapprentissage.

Nous comparons **deux tailles** et **deux valeurs de régularisation** par validation croisée à cinq plis, sur le train uniquement.

In [ ]:
validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=GRAINE)
mlp = MLPClassifier(
    activation="relu", solver="adam", learning_rate_init=.001,
    batch_size=32, max_iter=250, early_stopping=True,
    validation_fraction=.15, n_iter_no_change=14, random_state=GRAINE)
parametres = {
    "hidden_layer_sizes": [(64,), (128,32)],
    "alpha": [.001, .01]
}
recherche = GridSearchCV(mlp, parametres, cv=validation,
                          scoring="roc_auc", n_jobs=1)
recherche.fit(X_train, y_train)
modele = recherche.best_estimator_
scores_cv = pd.DataFrame(recherche.cv_results_)
cols = ["param_hidden_layer_sizes", "param_alpha", "mean_test_score", "std_test_score"]
scores_cv[cols].to_csv(SORTIE / "benchmark_mlp_cv.csv", index=False)
print("Meilleurs paramètres :", recherche.best_params_)
print("ROC-AUC CV :", round(recherche.best_score_, 3))
print(scores_cv[cols].round(3).to_string(index=False))

## 6. Visualiser les architectures et la courbe d'apprentissage

La **loss** mesure l'erreur d'apprentissage. La courbe de validation interne contient de l'**accuracy**, alors que le choix du meilleur réseau se fait avec la ROC-AUC en validation croisée.

In [ ]:
r = scores_cv.sort_values("mean_test_score")
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{h}, alpha={a}" for h,a in zip(r["param_hidden_layer_sizes"], r["param_alpha"])]
ax.barh(labels, r["mean_test_score"], xerr=r["std_test_score"],
        color="#5D9ABE", capsize=4)
ax.set_xlim(0,1)
ax.set_xlabel("ROC-AUC moyenne (CV)")
ax.set_title("Comparaison des réseaux")
fig.tight_layout()
fig.savefig(SORTIE / "03_reseaux_cv.png", dpi=160)
plt.show()

fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(modele.loss_curve_, color="#287FA5")
axes[0].set_title("Loss d'entraînement")
axes[0].set_xlabel("Époque")
if modele.validation_scores_ is not None:
    axes[1].plot(modele.validation_scores_, color="#399E86")
axes[1].set_title("Accuracy validation interne")
axes[1].set_xlabel("Époque")
fig.tight_layout()
fig.savefig(SORTIE / "04_apprentissage.png", dpi=160)
plt.show()
print("Époques :", modele.n_iter_)

## 7. Évaluer le meilleur réseau sur le test final

On utilise la probabilité d'inhibition `predict_proba` pour la ROC-AUC et la précision-rappel, puis un seuil de **0,5** pour la matrice de confusion et le F1-score. On n'optimise pas ce seuil sur le test.

In [ ]:
proba = modele.predict_proba(X_test)[:,1]
prediction = (proba >= .5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0,1]).ravel()
bilan = {
    "model": "MLP", "test_n": len(y_test),
    "roc_auc": roc_auc_score(y_test, proba),
    "average_precision": average_precision_score(y_test, proba),
    "accuracy": accuracy_score(y_test, prediction),
    "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
    "precision": precision_score(y_test, prediction, zero_division=0),
    "recall": recall_score(y_test, prediction, zero_division=0),
    "f1": f1_score(y_test, prediction, zero_division=0),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
}
print(pd.DataFrame([bilan]).round(3).to_string(index=False))
pd.DataFrame([bilan]).to_csv(SORTIE/"benchmark_mlp_test.csv", index=False)

dummy = DummyClassifier(strategy="prior").fit(X_train,y_train)
fig, axes = plt.subplots(1,2,figsize=(11,4.6))
RocCurveDisplay.from_predictions(y_test,proba,ax=axes[0],name="MLP")
RocCurveDisplay.from_predictions(y_test,dummy.predict_proba(X_test)[:,1],ax=axes[0],name="Dummy")
PrecisionRecallDisplay.from_predictions(y_test,proba,ax=axes[1],name="MLP")
PrecisionRecallDisplay.from_predictions(y_test,dummy.predict_proba(X_test)[:,1],ax=axes[1],name="Dummy")
axes[0].set_title("ROC test");axes[1].set_title("Précision-rappel test")
fig.tight_layout()
fig.savefig(SORTIE/"05_roc_pr.png",dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test,prediction,
    display_labels=["Non-inhib.","Inhib."],labels=[0,1],
    cmap="Blues",colorbar=False,ax=ax)
ax.set_title("Matrice de confusion du MLP")
fig.tight_layout()
fig.savefig(SORTIE/"06_confusion.png",dpi=160)
plt.show()

## 8. Export des résultats

Les prédictions par molécule sont utiles pour examiner les faux négatifs (inhibiteurs manqués). Le modèle est enregistré localement pour éviter de refaire l'entraînement.

In [ ]:
predictions = d.iloc[i_test][["compound_name","source","canonical_smiles","active"]].copy()
predictions["mlp_proba"] = proba
predictions["mlp_prediction"] = prediction
predictions["erreur"] = predictions["active"] != prediction
predictions.to_csv(SORTIE/"mlp_predictions_test.csv", index=False)
joblib.dump({"modele":modele,"radius":2,"fpSize":1024},SORTIE/"mlp_bsep.joblib")
print("Fichiers générés dans",SORTIE.resolve())

## 9. Interprétation biologique et comparaison au SVM

Lors d'un premier test sur le jeu Morgan–Warner, le **SVM RBF a obtenu ROC-AUC = 0,845 et rappel = 76,3 %** ; le MLP simple est évalué sur la même séparation pour permettre la comparaison. Si tu changes les données ou les partitions, ces scores SVM ne sont plus directement comparables.

Un réseau de neurones n'est **pas nécessairement meilleur** qu'un SVM avec seulement 660 molécules.

**Limites :** les sources Morgan et Warner ont des seuils expérimentaux différents ; la séparation aléatoire peut être optimiste pour de nouvelles structures chimiques ; l'inhibition de BSEP n'est pas toute la toxicité hépatique. Une séparation par scaffold serait une extension intéressante.

### Références
- McLoughlin et al. (2021), *Machine Learning Models to Predict Inhibition of the Bile Salt Export Pump*. https://doi.org/10.1021/acs.jcim.0c00950
- Dataset : https://github.com/ATOMScience-org/AMPL/tree/master/atomsci/ddm/examples/BSEP/data
- MLPClassifier : https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html